In [1]:
####################################
#ENVIRONMENT SETUP

In [2]:
#LIBRARIES
import os, sys

import numpy as np
import math

import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

# import xarray as xr
# import uxarray as ux

In [3]:
#Importing DirectoryManager Class

sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [4]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "SurfaceVariableAnimations_Unstructured"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/MPAS_Model_Data/InitialFigures



In [5]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class

In [6]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

[dataVariables, dataVariables_diag] = ModelData.GetVariableNames()
# dataVariables_diag


=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           MOIST
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-30 to 2022-07-03
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1', 'nSoilLevels']
 # History Files:208
 # Diag Files:   179
 # Time Steps:   289
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL
 Static File:    TRACER_regional5250_scaled3_x20.835586.static.latlon.nc



In [ ]:
###############
#FUNCTIONS

In [86]:
def LatLonBoundingBox_Center(campaign="TRACER"):
    if campaign == "TRACER":
        (latCenter, lonCenter) = 29.67, -95.059
    return latCenter,lonCenter

def LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500):
    # Earth radius in km
    R = 6371.0

    # Convert degrees to radians
    latRadians = math.radians(latCenter)

    # Calculate degree offsets
    dLat = (radius_km / R) * (180.0 / math.pi)
    dLon = (radius_km / (R * math.cos(latRadians))) * (180.0 / math.pi)

    # Bounding box
    latMin = latCenter - dLat
    latMax = latCenter + dLat
    lonMin = lonCenter - dLon
    lonMax = lonCenter + dLon

    latBounds = (latMin, latMax)
    lonBounds = (lonMin, lonMax)
    return latBounds, lonBounds

def LatLonBoundingBox_Subset(variable, latBounds, lonBounds):
    """
    Subset a structured (lat-lon) xarray DataArray or Dataset
    to a given latitude/longitude bounding box.
    """
    # Determine coordinate names (support latitude/lat, longitude/lon)
    lat_name = "latitude" if "latitude" in variable.coords else "lat"
    lon_name = "longitude" if "longitude" in variable.coords else "lon"

    # Handle reversed latitude (if decreasing)
    lat_vals = variable[lat_name].values
    if lat_vals[0] > lat_vals[-1]:
        lat_slice = slice(latBounds[1], latBounds[0])
    else:
        lat_slice = slice(latBounds[0], latBounds[1])

    lon_slice = slice(lonBounds[0], lonBounds[1])

    # Perform subset
    variableSubset = variable.sel({lat_name: lat_slice, lon_name: lon_slice})

    # Extract matching lat/lon arrays
    lat = variableSubset[lat_name].values
    lon = variableSubset[lon_name].values

    # print(f"Subset region: lat={latBounds}, lon={lonBounds}")
    # print(f"Subset shape: {variableSubset[lat_name].shape} × {variableSubset[lon_name].shape}")

    return variableSubset, lat, lon

In [ ]:
def GetVariable(varName):
    if varName in dataVariables:
        return data[varName]
    elif varName in dataVariables_diag:
        return data_diag[varName]
    elif varName == "greenfrac":
        return ModelData.staticData["greenfrac"].isel(nMonths=6)

def GetVariable_Subset(varName):  
    variable = GetVariable(varName)
    [latCenter,lonCenter] = LatLonBoundingBox_Center(campaign="TRACER")
    [latBounds, lonBounds] = LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=1000)
    variableSubset, lat, lon = LatLonBoundingBox_Subset(variable,latBounds, lonBounds)

    # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
    return variableSubset, lat, lon

In [63]:
#PlotVariable_with_Borders()

# Preload map features once
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def PlotVariable_with_Borders(variable, varName, lat,lon, outputFile=None, save=False, cmap="viridis", title=""):
    """
    Plot a uxarray or xarray variable on a map with coastlines, borders, and states,
    using Matplotlib (static PNG output). Works headlessly — no Selenium needed.
    """

    # Create figure
    fig, ax = plt.subplots(
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(9, 5)
    )

    matrix = variable.data
    # Scatter/contour fill (tricontourf works for unstructured grids)
    im = ax.contourf(
        lon, lat, matrix,
        levels=60,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
    )

    # Add map features
    ax.add_feature(COAST, linewidth=1)
    ax.add_feature(BORDERS, linewidth=0.8)
    ax.add_feature(STATES, linewidth=0.5)
    ax.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    ax.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Colorbar + title
    plt.colorbar(im, ax=ax, orientation="vertical", label=varName)
    ax.set_title(title or varName)

    # Save or display
    if save:
        plt.savefig(outputFile, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved static PNG: {outputFile}")
        return None
    else:
        return fig


In [90]:
def BuildVariableDictionary(varNames):
    variableDictionary = {}
    for varName in varNames:
        # print(f"Adding {varName}")

        # Getting Output Directory
        folderName = varName
        timeString = ModelData.timeStrings[t]
        fileName = f"{varName}_{timeString}.png"
        outputFile = DirectoryManager.GetOutputFile(outputDirectory, folderName, fileName)
        
        # Handle addition of two variables
        if '+' in varName:
            var1, var2 = varName.split('+')
            var1 = var1.strip()
            var2 = var2.strip()
            
            subset1, lat1, lon1 = GetVariable_Subset(var1)
            subset2, lat2, lon2 = GetVariable_Subset(var2)
    
            # Make sure lat/lon are compatible (e.g., same shape)
            if not (np.array_equal(lat1, lat2) and np.array_equal(lon1, lon2)):
                raise ValueError(f"Lat/lon mismatch for {var1} and {var2}")
    
            variableSubset = subset1 + subset2
            lat, lon = lat1, lon1
        else:
            variableSubset, lat, lon = GetVariable_Subset(varName)
    
        # Store in dictionary
        variableDictionary[varName] = {
            "data": variableSubset,
            "lat": lat,
            "lon": lon,
            "outputFile": outputFile
        }
        
    return variableDictionary

def MakePlots(variableDictionary):
    for varName, contents in variableDictionary.items():
        # print(f"Plotting {varName}")
    
        data = contents["data"]
        lat  = contents["lat"]
        lon  = contents["lon"]
        outputFile = contents["outputFile"]
        
        fig = PlotVariable_with_Borders(data, varName, lat, lon, outputFile, save=True, cmap="viridis")

In [87]:
#################
#RUNNING

In [ ]:
#time loop
# num_times = ModelData.NTime
num_times = 100
for t in range(num_times):
    if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
    #loading data
    data = ModelData.GetDataTimestep(t)
    data_diag = ModelData.GetDataTimestep_diag(t); print(f"\n")

    #defining variable names
    varNames = [
        "surface_pressure",
        "u10", "v10", "q2",
        "hfx", "qfx", "lh",
        "rainnc+rainc",
        "refl10cm_1km"
    ] + (["greenfrac"] if t == 0 else [])

    # running
    variableDictionary = BuildVariableDictionary(varNames)
    MakePlots(variableDictionary)

    break

In [ ]:
# xlabel, ylabel, units, title (including time and case location)
# also need clims